# GNN fixed-architecture runs (one operator, seven feature configurations)

Runs on a Kaggle GPU notebook: one operator, the seven feature configurations, the fixed hyperparameters (`conf/hyperparameters/gnn/fixed.yaml`).
- Settings (right panel): Accelerator = GPU, Internet = On.
- Input: the dataset `sandrokhizanishvili/hi-small-gnn` (graphs + feature_meta.json).
- Set `MODEL` below, then Run All (or Save Version -> Save & Run All to run headless, 12 h limit).
- Each run writes `results.json`, `history.csv`, `curves.png`, `predictions.csv`, `best.pt`, `config.yaml` to `/kaggle/working/outputs/train/<model>/<run>/seed-42/`.

In [ ]:
MODEL = 'gin'            # gin | pna | gat | transformer
FEATURE_CONFIGS = 'c1,c2,c3,c4,c5,c6,c7'
SEED = 42

In [ ]:
# The pinned torch / PyG stack known to work on Kaggle GPU; do not change versions.
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch_geometric lightgbm==4.6.0 hydra-core==1.3.2

In [ ]:
# The code: clone once, pull on a rerun.
import os
if os.path.isdir('/kaggle/working/gnn_for_aml'):
    !cd /kaggle/working/gnn_for_aml && git pull
else:
    !git clone https://github.com/sandrokhizanishvili/gnn_for_aml.git /kaggle/working/gnn_for_aml
%cd /kaggle/working/gnn_for_aml
!ls /kaggle/input/datasets/sandrokhizanishvili/hi-small-gnn

In [ ]:
# Environment test: versions, the GPU, one forward / backward pass per operator.
!python run.py mode=check machine=kaggle

In [ ]:
# The seven runs, one after another. A finished run is skipped, so the cell can be rerun after an interruption.
!python run.py -m model={MODEL} feature_config={FEATURE_CONFIGS} seed={SEED} machine=kaggle

In [ ]:
# The metrics of every finished run of this operator, one row each.
import glob, json, pandas as pd
rows = [json.load(open(p)) for p in sorted(glob.glob(f'/kaggle/working/outputs/train/{MODEL}/*/seed-*/results.json'))]
cols = ['run', 'best_epoch', 'threshold', 'params', 'invariant_params'] + [f'{s}_{m}' for s in ('train', 'val', 'test') for m in ('f1', 'precision', 'recall', 'pr_auc')]
pd.DataFrame(rows)[cols].round(4) if rows else "no finished run yet"

In [ ]:
# A small archive for download: everything except the checkpoints and predictions (the Output tab has the full folder).
!cd /kaggle/working && rm -f results_small.zip && zip -qr results_small.zip outputs -x "*.pt" "*predictions.csv" && ls -lh results_small.zip

## After the run
- Output tab: download `results_small.zip` (metrics, curves, histories) or the whole `outputs` folder (adds `predictions.csv` and `best.pt`, about 1.6 GB for seven runs).
- Unzip into `gnn_for_aml/outputs/` on the laptop; commit the small files (`predictions.csv` and `*.pt` are gitignored); mark the rows in `EXPERIMENTS.md`.
- Next operator: change `MODEL` and run again in a new session.